In [10]:
from pathlib import Path
import shutil
import pandas as pd
import cv2

from sklearn.model_selection import GroupShuffleSplit, GroupKFold


# -------------------------
# Config
# -------------------------
INPUT_DF = "/export/home/rstanciu/FM_thesis_Razvan/FM_thesis/df_masses.pkl"
OUT_ROOT = Path("/export/home/rstanciu/ZGT_Mammo-FM_format")

ROOT = "/mnt/data/spathak/"

IMAGE_COL = "ImagePath"
MASK_COL = "ROIPath"
PATIENT_COL = "PatientID"
VIEW_COL = "View"
ROI_COL = "ROINum"

N_FOLDS = 5
TEST_SIZE = 0.2
RANDOM_STATE = 42

CLASS_NAME = "Mass"


# -------------------------
# Helpers
# -------------------------
def clean_token(x):
    if pd.isna(x) or str(x).lower() in ["none", "nan", ""]:
        return "none"
    return (
        str(x)
        .replace("/", "_")
        .replace("\\", "_")
        .replace(" ", "_")
    )


def safe_join(root, path):
    path = str(path)
    if path.startswith("/"):
        return Path(path)
    return Path(root) / path


def mask_to_bbox(mask_path):
    mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)

    if mask is None:
        raise ValueError(f"Could not read mask: {mask_path}")

    ys, xs = (mask > 0).nonzero()

    if len(xs) == 0 or len(ys) == 0:
        return -1, -1, -1, -1

    return int(xs.min()), int(ys.min()), int(xs.max()), int(ys.max())


def read_image_size(image_path):
    img = cv2.imread(str(image_path), cv2.IMREAD_GRAYSCALE)

    if img is None:
        raise ValueError(f"Could not read image: {image_path}")

    h, w = img.shape[:2]
    return h, w


def copy_if_needed(src, dst):
    dst.parent.mkdir(parents=True, exist_ok=True)
    if not dst.exists():
        shutil.copy2(src, dst)


# -------------------------
# Load original dataframe
# -------------------------
df = pd.read_pickle(INPUT_DF)

OUT_ROOT.mkdir(parents=True, exist_ok=True)
(OUT_ROOT / "images_png").mkdir(exist_ok=True)
(OUT_ROOT / "masks").mkdir(exist_ok=True)

rows = []


# -------------------------
# Copy images/masks and create detection rows
# -------------------------
for idx, row in df.iterrows():
    image_path = safe_join(ROOT, row[IMAGE_COL])

    if pd.isna(row[MASK_COL]):
        mask_path = None
    else:
        mask_path = safe_join(ROOT, row[MASK_COL])

    patient = clean_token(row[PATIENT_COL])
    view = clean_token(row[VIEW_COL])
    roi = clean_token(row[ROI_COL])

    # Detection image ID should identify the mammogram,
    # not the individual ROI.
    image_id = f"{patient}_{view}"

    # ROI-specific ID for masks/debugging.
    unique_id = f"{patient}_{view}_{roi}"

    dst_patient_img_dir = OUT_ROOT / "images_png" / patient
    dst_patient_mask_dir = OUT_ROOT / "masks" / patient

    dst_img_name = f"{image_id}.png"
    dst_mask_name = f"{unique_id}.png"

    dst_img_path = dst_patient_img_dir / dst_img_name
    dst_mask_path = dst_patient_mask_dir / dst_mask_name

    copy_if_needed(image_path, dst_img_path)

    h, w = read_image_size(dst_img_path)

    has_mask = mask_path is not None and mask_path.exists()

    if has_mask:
        copy_if_needed(mask_path, dst_mask_path)
        x_min, y_min, x_max, y_max = mask_to_bbox(dst_mask_path)
        class_name = CLASS_NAME if x_min >= 0 else "No finding"
        rel_mask_path = str(Path(patient) / dst_mask_name)
    else:
        x_min, y_min, x_max, y_max = -1, -1, -1, -1
        class_name = "No finding"
        rel_mask_path = ""

    rows.append({
        "patient_id": patient,
        "image_id": image_id,
        "unique_id": unique_id,
        "view": view,
        "roi_num": roi,

        # Relative path from images_png/
        "image_path": str(Path(patient) / dst_img_name),

        # Optional/debug only; detector usually only needs boxes.
        "mask_path": rel_mask_path,

        "class_name": class_name,
        "x_min": x_min,
        "y_min": y_min,
        "x_max": x_max,
        "y_max": y_max,
        "height": h,
        "width": w,

        "original_image_path": str(image_path),
        "original_mask_path": str(mask_path) if mask_path is not None else "",
    })


new_df = pd.DataFrame(rows)


# -------------------------
# Patient-wise train/test split
# -------------------------
new_df["split"] = "trainval"
new_df["fold"] = -1

groups = new_df["patient_id"]

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
)

trainval_idx, test_idx = next(gss.split(new_df, groups=groups))

new_df.loc[test_idx, "split"] = "test"
new_df.loc[trainval_idx, "split"] = "trainval"


# -------------------------
# Patient-wise k-folds inside trainval only
# -------------------------
trainval_df = new_df.loc[trainval_idx].copy()
trainval_groups = trainval_df["patient_id"]

gkf = GroupKFold(n_splits=N_FOLDS)

for fold, (_, val_idx_local) in enumerate(
    gkf.split(trainval_df, groups=trainval_groups)
):
    val_indices = trainval_df.iloc[val_idx_local].index
    new_df.loc[val_indices, "fold"] = fold


# -------------------------
# Save CSV
# -------------------------
out_csv = OUT_ROOT / "detection_folds.csv"
new_df.to_csv(out_csv, index=False)


# -------------------------
# Sanity checks
# -------------------------
print(f"Saved prepared dataset to: {OUT_ROOT}")
print(f"Saved CSV to: {out_csv}")

print("\nSplit counts:")
print(new_df.groupby("split")["patient_id"].nunique())

print("\nFold counts inside trainval:")
print(
    new_df[new_df["split"] == "trainval"]
    .groupby("fold")["patient_id"]
    .nunique()
)

print("\nRows per split:")
print(new_df["split"].value_counts())

print("\nExample rows:")
print(new_df.head())

Saved prepared dataset to: /export/home/rstanciu/ZGT_Mammo-FM_format
Saved CSV to: /export/home/rstanciu/ZGT_Mammo-FM_format/detection_folds.csv

Split counts:
split
test        14
trainval    53
Name: patient_id, dtype: int64

Fold counts inside trainval:
fold
0    10
1    10
2    11
3    11
4    11
Name: patient_id, dtype: int64

Rows per split:
split
trainval    125
test         34
Name: count, dtype: int64

Example rows:
  patient_id image_id   unique_id   view roi_num     image_path  \
0          1   1_L_CC   1_L_CC_R1   L_CC      R1   1/1_L_CC.png   
1          3   3_L_CC   3_L_CC_R1   L_CC      R1   3/3_L_CC.png   
2          3  3_L_MLO  3_L_MLO_R1  L_MLO      R1  3/3_L_MLO.png   
3          4   4_R_CC   4_R_CC_R2   R_CC      R2   4/4_R_CC.png   
4          4  4_R_MLO  4_R_MLO_R2  R_MLO      R2  4/4_R_MLO.png   

          mask_path class_name  x_min  y_min  x_max  y_max  height  width  \
0   1/1_L_CC_R1.png       Mass    360   2123    579   2237    2921   1283   
1   3/3_L_CC_R

In [9]:
new_df.head()

,patient_id,image_id,unique_id,view,roi_num,image_path,mask_path,class_name,x_min,y_min,x_max,y_max,height,width,original_image_path,original_mask_path,split,fold
0,1,1_L_CC,1_L_CC_R1,L_CC,R1,1/1_L_CC.png,1/1_L_CC_R1.png,Mass,360,2123,579,2237,2921,1283,/mnt/data/spathak/CLaM-Annot/1.2.826.0.1.36800...,/mnt/data/spathak/CLaM-Annot/1.2.826.0.1.36800...,test,-1
1,3,3_L_CC,3_L_CC_R1,L_CC,R1,3/3_L_CC.png,3/3_L_CC_R1.png,Mass,89,1737,308,1956,2980,1191,/mnt/data/spathak/CLaM-Annot/1.2.826.0.1.36800...,/mnt/data/spathak/CLaM-Annot/1.2.826.0.1.36800...,trainval,0
2,3,3_L_MLO,3_L_MLO_R1,L_MLO,R1,3/3_L_MLO.png,3/3_L_MLO_R1.png,Mass,0,1847,107,2133,2791,1318,/mnt/data/spathak/CLaM-Annot/1.2.826.0.1.36800...,/mnt/data/spathak/CLaM-Annot/1.2.826.0.1.36800...,trainval,0
3,4,4_R_CC,4_R_CC_R2,R_CC,R2,4/4_R_CC.png,4/4_R_CC_R2.png,Mass,1109,492,1525,884,3131,1605,/mnt/data/spathak/CLaM-Annot/1.2.826.0.1.36800...,/mnt/data/spathak/CLaM-Annot/1.2.826.0.1.36800...,trainval,4
4,4,4_R_MLO,4_R_MLO_R2,R_MLO,R2,4/4_R_MLO.png,4/4_R_MLO_R2.png,Mass,957,274,1488,748,2797,1716,/mnt/data/spathak/CLaM-Annot/1.2.826.0.1.36800...,/mnt/data/spathak/CLaM-Annot/1.2.826.0.1.36800...,trainval,4


In [16]:
for fold in range(5):
    fold_df = new_df[new_df["split"] == "trainval"].copy()

    fold_df["split"] = fold_df["fold"].apply(
        lambda x: "test" if x == fold else "training"
    )

    fold_df.to_csv(OUT_ROOT / f"detection_mammofm_fold{fold}.csv", index=False)

# untouched final test set
final_test = new_df[new_df["split"] == "test"].copy()
final_test.to_csv(OUT_ROOT / "detection_final_test.csv", index=False)

In [19]:
print(new_df)


    patient_id   image_id     unique_id    view roi_num         image_path  \
0            1     1_L_CC     1_L_CC_R1    L_CC      R1       1/1_L_CC.png   
1            3     3_L_CC     3_L_CC_R1    L_CC      R1       3/3_L_CC.png   
2            3    3_L_MLO    3_L_MLO_R1   L_MLO      R1      3/3_L_MLO.png   
3            4     4_R_CC     4_R_CC_R2    R_CC      R2       4/4_R_CC.png   
4            4    4_R_MLO    4_R_MLO_R2   R_MLO      R2      4/4_R_MLO.png   
..         ...        ...           ...     ...     ...                ...   
154         98  98_R_XCCL  98_R_XCCL_R1  R_XCCL      R1   98/98_R_XCCL.png   
155         99    99_R_CC    99_R_CC_R1    R_CC      R1     99/99_R_CC.png   
156         99   99_R_MLO   99_R_MLO_R1   R_MLO      R1    99/99_R_MLO.png   
157        100   100_L_CC   100_L_CC_R1    L_CC      R1   100/100_L_CC.png   
158        100  100_L_MLO  100_L_MLO_R1   L_MLO      R1  100/100_L_MLO.png   

                mask_path class_name  x_min  y_min  x_max  y_ma